# Modelo de Optimizacion para modem 4G

Instalacion paquete JuMP

In [1]:
import Pkg # Importa el administrador de paquetes Pkg
Pkg.add("JuMP") # Instala el paquete JuMP

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


Instalacion de Solver

In [2]:
Pkg.add("GLPK") # Instala el paquete GLPK

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [3]:
using JuMP
using GLPK

# Crear el modelo de optimización usando el solver GLPK
model = Model(GLPK.Optimizer)

# Desactivar la salida de logs en la terminal
set_silent(model)

# Número total de máquinas
n_nodes = 21

# Definición de Variables: x[i] es 1 si se instala un módem, 0 en caso contrario
@variable(model, x[1:n_nodes], Bin)

# 4. Construcción de la Topología del Grafo (Vecindarios)
N = [Set([i]) for i in 1:n_nodes]

function add_edge!(N, u, v)
    push!(N[u], v)
    push!(N[v], u)
end

# --- Componente Superior (Nodos 2 al 7) ---
edges_top = [
    (2,3), (2,4), (2,6), (2,7),
    (3,4), (3,5), (3,6),
    (4,5), (4,6),
    (5,6),
    (6,7)
]
for (u, v) in edges_top
    add_edge!(N, u, v)
end

# --- Componente Inferior: Cola (Nodos 9 al 12) ---
edges_tail = [
    (9,10), (9,12),
    (10,11), (10,12),
    (11,12)
]
for (u, v) in edges_tail
    add_edge!(N, u, v)
end

# --- Conexión entre la Cola y el Núcleo Denso ---
edges_bridge = [
    (12,13), (12,14), (12,15)
]
for (u, v) in edges_bridge
    add_edge!(N, u, v)
end

# --- Componente Inferior: Núcleo Denso (Nodos 13 al 21) ---
edges_core = [
    (13,14), (13,15), (13,16), (13,18), (13,19), (13,20), (13,21),
    (14,15), (14,16), (14,17), (14,18), (14,19), (14,20), (14,21),
    (15,16), (15,17), (15,18), (15,19), (15,20), (15,21),
    (16,17), (16,18), (16,19), (16,20), (16,21),
    (17,18), (17,19), (17,20), (17,21),
    (18,19), (18,20), (18,21),
    (20,21)
]
for (u, v) in edges_core
    add_edge!(N, u, v)
end

# 5. Función Objetivo: Minimizar el número total de módems
@objective(model, Min, sum(x[i] for i in 1:n_nodes))

# 6. Restricciones del Modelo

# A) Nodos Aislados (1 y 8): Mínimo 1 módem
@constraint(model, sum(x[j] for j in N[1]) >= 1)
@constraint(model, sum(x[j] for j in N[8]) >= 1)

# B) Nodos Conectados (del 2 al 21 excepto el 8): Redundancia mínima de 2
for i in 1:n_nodes
    if i != 1 && i != 8
        @constraint(model, sum(x[j] for j in N[i]) >= 2)
    end
end

# 7. Resolver el problema
optimize!(model)

# 8. Reporte de Resultados
println("==========================================")
println(" Estado de la Solución: ", termination_status(model))
println(" Total de Módems Mínimo: ", Int(objective_value(model)))
println("==========================================")
println("Instalar módems en las siguientes máquinas:")

for i in 1:n_nodes
    if value(x[i]) > 0.5
        println("  -> Máquina x[$i]")
    end
end

 Estado de la Solución: OPTIMAL
 Total de Módems Mínimo: 9
Instalar módems en las siguientes máquinas:
  -> Máquina x[1]
  -> Máquina x[2]
  -> Máquina x[3]
  -> Máquina x[6]
  -> Máquina x[8]
  -> Máquina x[10]
  -> Máquina x[12]
  -> Máquina x[14]
  -> Máquina x[15]
